In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# Euler — well-observed degraders by 100%-SoH start month (BMS-capacity SoH + condition-aware model)

Euler counterpart of the Mahindra analysis in `predictions_june2023`. SoH = validated **BMS-capacity**;
forecast = condition-aware Euler model rolled to the 80% EoFL. Grouped by the month each vehicle's SoH
curve **starts at 100%** (battery new) — *not* registration date — so every curve in a plot shares the
age-axis origin (age since 100% SoH) and the calendar top axis is exact. Euler warranty is uniformly
**5 yr** (no 3/5 split), but the degradation-cause tag is richer — it adds **thermal**, **deep-DoD** and
**imbalance** (cell-imbalance), which Euler reports and Mahindra didn't. Legend = `vehicle · cause · pp-lost`.

In [ ]:
import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.express as px
from collections import Counter
import euler_features, euler_model as em, config
PAL = px.colors.qualitative.Dark24

if not Path('data/euler/features/feature_table.parquet').exists():
    euler_features.main()
meu = pd.read_parquet('data/euler/features/feature_table.parquet').sort_values(['vin', 'month'])
emodel = em.train(em.build_transitions(meu))

ereg = pd.read_csv('data/euler/Euler_Regd_Details.csv')
ereg['reg'] = pd.to_datetime(ereg['regd_date'], format='%d/%m/%y', errors='coerce')
EREG = dict(zip(ereg['vin'], ereg['reg']))
WYR = config.warranty_for('euler', '')[0]     # 5 yr for all Euler

def soh_at_age(d, age):
    A = np.concatenate([d['a_obs'], d['a_fc']]); S = np.concatenate([d['s_obs'], d['s_fc']])
    return float(np.interp(age, A, S))

T100 = {v: meu[meu['vin'] == v]['month'].min() for v in meu['vin'].unique()}   # month the SoH curve starts at ~100% (battery new)
startm = {v: T100[v].to_period('M') for v in meu['vin'].unique()}              # GROUP KEY = 100%-SoH start month (= age-axis origin)
nmo = meu.groupby('vin').size()
dec = meu.groupby('vin')['soh'].apply(lambda s: float(s.iloc[0] - s.iloc[-1]))

# concise degradation driver: z-score recent operating stress vs the Euler fleet
_rec = pd.DataFrame({v: g.sort_values('month').iloc[-6:][em.STRESS].median() for v, g in meu.groupby('vin')}).T
_m = pd.DataFrame({'chg': _rec['cur_chg_mean'], 'hiSoC': _rec['frac_soc_high'], 'dis': _rec['cur_dis_mean'].abs(),
                   'pk': _rec['cur_abs_p95'], 'km': _rec['km_month'], 'temp': _rec['temp_max'],
                   'loSoC': _rec['frac_soc_low'], 'dod': _rec['dod_mean'], 'imb': _rec['imbalance_mean']})
_z = (_m - _m.mean()) / _m.std(ddof=0).replace(0, 1)
_grp = {'charging': ['chg', 'hiSoC'], 'driving': ['dis', 'pk', 'km'], 'thermal': ['temp'],
        'deep-DoD': ['loSoC', 'dod'], 'imbalance': ['imb']}
_gz = pd.DataFrame({g: _z[c].mean(axis=1) for g, c in _grp.items()})
def deg_tag(vin):
    gz = _gz.loc[vin]
    return gz.idxmax() if gz.max() >= 0.5 else 'age'

def fc_to_eol(g, eol=80.0, max_age=8.0):
    last_age = float(g['age_months'].iloc[-1]); H = max(int(round(max_age * 12 - last_age)), 1)
    fc = np.array(em.free_run(g, emodel, H))
    if (fc <= eol).any():
        fc = fc[:int(np.argmax(fc <= eol)) + 1]
    return fc

def eu_d(vin):
    g = meu[meu['vin'] == vin].sort_values('month'); t100 = T100[vin]       # anchor age at the 100%-SoH month
    s_obs = g['soh'].to_numpy(); a_obs = ((g['month'] - t100).dt.days / 365.25).to_numpy()
    fc = fc_to_eol(g)
    a_fc = np.array([((g['month'].iloc[-1] + pd.DateOffset(months=k)) - t100).days / 365.25 for k in range(1, len(fc) + 1)])
    return dict(vin=vin, reg=EREG.get(vin), t100=t100, a_obs=a_obs, s_obs=s_obs, a_fc=a_fc, s_fc=fc, wyr=WYR,
                label=f"{vin[-6:]} now {s_obs[-1]:.0f}%")
print(f'{meu.vin.nunique()} Euler vehicles in feature table; warranty {WYR} yr')

## Well-observed degraders per registration month

In [ ]:
MIN_MONTHS = 15     # "good amount of data points"
MIN_DECLINE = 7     # "not flat" — total observed SoH drop (pp)
N_SHOW = 6

def show_month(period):
    vins = sorted([v for v in meu['vin'].unique()
                   if startm[v] == period and nmo[v] >= MIN_MONTHS and dec[v] >= MIN_DECLINE],
                  key=lambda v: dec[v], reverse=True)[:N_SHOW]
    if len(vins) < 2:
        return
    sel = [eu_d(v) for v in vins]
    print(f'100% SoH from {period}: {len(sel)} ->', [(d['vin'][-6:], f"-{dec[d['vin']]:.0f}pp", deg_tag(d['vin'])) for d in sel])
    ref = pd.Timestamp(int(np.median([pd.Timestamp(d['t100']).value for d in sel])))
    ymin = min(min(d['s_obs'].min(), d['s_fc'].min()) for d in sel) - 2
    xmax = max(max(d['a_fc'][-1] for d in sel), WYR) + 0.3
    fig = go.Figure()
    for i, d in enumerate(sel):
        col = PAL[i % len(PAL)]; tail = d['vin'][-6:]; risk = soh_at_age(d, WYR) < 78
        fig.add_scatter(x=[0, d['a_obs'][0]], y=[100, d['s_obs'][0]], mode='lines', line=dict(color=col, width=1, dash='dot'),
                        opacity=0.5, legendgroup=tail, showlegend=False, hoverinfo='skip')
        fig.add_scatter(x=d['a_obs'], y=d['s_obs'], mode='lines+markers', line=dict(color=col, width=1.8),
                        marker=dict(size=6, color=col), name=f"{tail} · {soh_at_age(d, WYR):.0f}% @ {WYR}yr" + (' ✓' if soh_at_age(d, WYR) >= 80 else ''),
                        legendgroup=tail, hovertemplate=tail + ' actual %{y:.1f}%<extra></extra>')
        fig.add_scatter(x=d['a_fc'], y=d['s_fc'], mode='lines', line=dict(color=col, width=2.2, dash='dash'),
                        legendgroup=tail, showlegend=False, hovertemplate=tail + ' forecast %{y:.1f}%<extra></extra>')
        if d['s_fc'][-1] <= 80.5:
            fig.add_scatter(x=[d['a_fc'][-1]], y=[d['s_fc'][-1]], mode='markers+text', legendgroup=tail, showlegend=False,
                            marker=dict(symbol='circle-open', size=9, color=col, line=dict(width=2)),
                            text=[f"{d['a_fc'][-1]:.1f}yr"], textposition='bottom center', textfont=dict(size=9, color=col),
                            hovertemplate=tail + ' hits 80%% at %{x:.1f} yr<extra></extra>')
        fig.add_scatter(x=[WYR], y=[soh_at_age(d, WYR)], mode='markers', legendgroup=tail, showlegend=False,
                        marker=dict(symbol='triangle-down', size=11, color=('crimson' if risk else 'seagreen'), line=dict(color='black', width=0.6)),
                        hovertemplate=tail + f" @{WYR}yr warranty " + '%{y:.1f}%<extra></extra>')
    fig.add_hline(y=80, line=dict(color='red', width=1.3, dash='dot'), annotation_text='80% EoFL', annotation_position='bottom left')
    fig.add_vline(x=WYR, line=dict(color='dimgray', width=1.3, dash='dashdot'))
    fig.add_annotation(x=WYR, y=ymin + 1.5, text=f'{WYR}-yr warranty', showarrow=False, textangle=-90,
                       xanchor='right', yanchor='bottom', font=dict(color='dimgray', size=10))
    ages = list(range(0, int(np.ceil(xmax)) + 1))
    dates = [(ref + pd.Timedelta(days=a * 365.25)).strftime('%b %Y') for a in ages]
    fig.add_scatter(x=ages, y=[ymin] * len(ages), xaxis='x2', mode='markers', marker=dict(opacity=0), showlegend=False, hoverinfo='skip')
    fig.update_layout(height=540, width=1120, template='plotly_white', margin=dict(t=70),
                      title=dict(text=f"Euler — 100% SoH from {period}: well-observed degraders (legend = vehicle · SoH retained @ warranty)", y=0.97),
                      xaxis=dict(title='age since 100% SoH (years)', range=[0, xmax]),
                      xaxis2=dict(overlaying='x', side='top', range=[0, xmax], tickmode='array', tickvals=ages, ticktext=dates,
                                  title=dict(text='calendar (year-month)', standoff=6), tickfont=dict(size=9), title_font=dict(size=10)),
                      yaxis=dict(title='BMS-capacity SoH (%)', range=[ymin, 101]),
                      legend=dict(title='vehicle · SoH @ warranty', x=1.01, y=1, font=dict(size=10)))
    fig.show()

qual = [v for v in meu['vin'].unique() if nmo[v] >= MIN_MONTHS and dec[v] >= MIN_DECLINE]
months = sorted([p for p, c in Counter(startm[v] for v in qual).items() if p.year in (2023, 2024) and c >= 2])
print('100%-SoH start months with >=2 well-observed degraders:', [str(p) for p in months])
for p in months:
    show_month(p)